In [1]:
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from pathlib import Path

import mediapipe as mp

import numpy as np
import ffmpeg
import logging
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
print(hasattr(ffmpeg, "input"))
PARENT_PATH = Path(os.getenv("PARENT_DIR", "."))  # default fallback to current directory

if not os.path.exists(PARENT_PATH / "src" / "logs"):
    os.mkdir(PARENT_PATH / "src" / "logs")
    
logging.basicConfig(
    filename = PARENT_PATH / "src" / "logs" / "app.log",
    filemode="a",  # append mode
    level=logging.DEBUG,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
file_handler = logging.FileHandler("app.log")
logger.addHandler(file_handler)

True


In [3]:
data_path = PARENT_PATH / "data"
in_file_path = data_path / "input_videos" / "input_vid1.mp4"
output_audio_path = data_path / "output_audios"
output_video_path = data_path / "output_videos"
# Extract audio and video streams
input_file = ffmpeg.input(in_file_path)

audio_stream = input_file.audio
video_stream = input_file.video

audio_only_file = "audio_only.mp3"
video_only_file = "video_only.mp4"

audio_path = Path(output_audio_path / audio_only_file)
video_path = Path(output_video_path / video_only_file)

try:
    if audio_path.exists():
        logger.error(f"{audio_path} already exists. Aborting file creation.")
    else:
        ffmpeg.output(audio_stream, str(audio_path)).run()
        logger.info(f"{audio_only_file} created successfully")
    if video_path.exists():
        logger.error(f"{video_path} already exists. Aborting file creation.")
    else:
        ffmpeg.output(video_stream, str(video_path)).run()
        logger.info(f"{video_only_file} created successfully")

except IOError as e:
    # Handle other potential I/O errors (e.g., permissions issues)
    logger.error(f"An I/O error occurred: {e}")


In [4]:
# visual_loader.py


# -----------------------------
# SAFE FACE VALIDATION
# -----------------------------
def _is_valid_face(face):
    return (
        face is not None
        and isinstance(face, np.ndarray)
        and face.size > 0
        and face.shape[0] > 10    # height
        and face.shape[1] > 10    # width
    )

# -----------------------------
# SAFE FACE EXTRACTOR
# -----------------------------
def extract_face(frame):
    """Safe face extractor: mediapipe instance each call."""
    with mp.solutions.face_detection.FaceDetection(
        model_selection=1,
        min_detection_confidence=0.5
    ) as fd:

        if frame is None or frame.size == 0:
            return None

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = fd.process(rgb)

        if not results.detections:
            return None

        det = results.detections[0]
        bbox = det.location_data.relative_bounding_box

        h, w, _ = frame.shape
        x1 = int(bbox.xmin * w)
        y1 = int(bbox.ymin * h)
        x2 = int((bbox.xmin + bbox.width) * w)
        y2 = int((bbox.ymin + bbox.height) * h)

        face = frame[y1:y2, x1:x2]

        if not _is_valid_face(face):
            return None

        return face

# ===========================================================
#                CELEB-DF VISUAL DATASET
# ===========================================================
class CelebDFVisualDataset(Dataset):
    def __init__(self, root_dir, frames_per_video=8, video_list= None):
        self.root_dir = Path(root_dir)
        self.frames_per_video = frames_per_video
        
        self.video_paths = []
        self.labels = []

        if video_list is None:
            # label mapping
            
            videos = list(self.root_dir.glob("*.mp4"))
            self.video_paths = videos
       
        
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])
    
    def __len__(self):
        return len(self.video_paths)

    # -----------------------------
    # SAFE VIDEO FRAME READING
    # -----------------------------
    def _read_video_frames(self, path):
        cap = cv2.VideoCapture(path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames < 1:
            cap.release()
            return torch.zeros(self.frames_per_video, 3, 224, 224)

        indices = torch.linspace(0, total_frames - 1, self.frames_per_video).long()
        frames = []

        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx.item()))
            ret, frame = cap.read()

            if not ret or frame is None:
                continue

            face = extract_face(frame)

            # -----------------------------
            # SAFETY CHECK
            # -----------------------------
            if not _is_valid_face(face):
                continue

            # now safe to convert
            face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

            img = self.transform(face)

            if img.shape[0] != 3:
                img = img[:3, :, :]

            frames.append(img)

        cap.release()

        # if nothing extracted → fallback
        if len(frames) == 0:
            return torch.zeros((self.frames_per_video, 3, 224, 224))

        # pad to fixed length
        while len(frames) < self.frames_per_video:
            frames.append(frames[-1].clone())

        return torch.stack(frames)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        frames = self._read_video_frames(video_path)
        # label = torch.tensor(self.labels[idx], dtype=torch.long)
        return frames #, label

In [5]:
frames_per_video = 8  # Number of frames to extract per video
dataset = CelebDFVisualDataset(
    root_dir=output_video_path,
    frames_per_video=frames_per_video
)

print (f"number of videos in dataset: {len(dataset)}")

number of videos in dataset: 1


In [7]:
print (f"path of output video: {dataset[0].shape}")

path of output video: torch.Size([8, 3, 224, 224])
